---
title: Memuat Data
short_title: Memuat data
subject: Panduan Pemula
subtitle: Memuat citra satelit dari datacube dengan dc.load.
description: Memuat citra satelit dari datacube dengan dc.load.
keywords:
  - open-data-cube
  - odc
  - beginner-guide
---
---
Notebook ini memperkenalkan `dc.load`, yaitu fungsi untuk mengambil citra satelit dari datacube.
Materi dimulai dengan penyusunan kueri, dilanjutkan dengan pembacaan data hasil pemuatan, lalu penerapan kueri tersebut pada wilayah lain.[^edits]

[^edits]: Notebook tutorial diperbarui secara otomatis sehingga perubahan dapat tertimpa saat pembaruan berlangsung.
    Simpan salinan kerja dalam file terpisah agar perubahan tetap tersimpan.

## A. Tujuan

- Memuat data dengan `dc.load`
- Membaca `xarray.Dataset` hasil pemuatan
- Menemukan dataset ODC sebelum memuat data piksel
- Memuat kumpulan dataset ODC yang telah dipilih

## B. Memuat data

### 1. Membuat koneksi ke datacube

Semua operasi dalam notebook ini diawali dengan koneksi ke datacube.
Objek `Datacube` tersebut disimpan dalam variabel `dc`, nama singkat yang lazim digunakan meskipun nama variabel lain juga tetap dapat dipakai.

In [ ]:
from datacube import Datacube

dc = Datacube(app="loading_data")

### 2. Menyusun kueri untuk `dc.load`

Masukkan pilihan produk, wilayah, waktu, measurement, dan grid keluaran ke dalam satu dictionary Python.
Dengan cara ini, semua nilai dapat diperiksa dan diubah bersama sebelum data piksel dicari dan dimuat.
Gunakan nama kunci yang sama dengan argumen yang diterima oleh `dc.load`, seperti pada kueri berikut.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": "2024",
    "measurements": ["red", "green", "blue"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

Kueri tersebut menggunakan parameter berikut:

- **product**: produk yang akan dimuat dari datacube.
  `dc.list_products()` menampilkan daftar lengkap produk yang tersedia.
- **x**, **y**: rentang bujur dan lintang dalam derajat yang masing-masing ditulis sebagai tuple berisi dua nilai.
  Rentang tersebut harus berpotongan dengan cakupan spasial produk; jika tidak, kueri tidak menghasilkan data.
- **time**: tanggal atau rentang tanggal.
  Nilainya dapat berupa tahun (`"2024"`), bulan (`"2024-01"`), tanggal tertentu (`"2024-01-15"`), atau tuple seperti `("2024-01-01", "2024-06-30")`.
- **measurements**: daftar measurement produk yang akan dikembalikan.
  `dc.list_measurements()` menunjukkan measurement yang tersedia pada setiap produk.
- **output_crs**: sistem referensi koordinat untuk grid hasil pemuatan, yang di sini ditulis sebagai kode EPSG `"EPSG:32647"` untuk UTM zona 47N.
  Parameter ini wajib diberikan jika produk tidak memiliki CRS keluaran bawaan, dan zona UTM setempat merupakan pilihan yang umum.
- **resolution**: ukuran piksel dalam satuan `output_crs`, dengan urutan `(y, x)`.
  Nilai y biasanya negatif karena baris array bergerak dari utara ke selatan.

Beberapa parameter lain tersedia untuk kebutuhan pemuatan tertentu:

- **datasets**: daftar objek `Dataset` ODC yang dikembalikan oleh `dc.find_datasets()`.
  Parameter ini digunakan dalam alur pencarian lalu pemuatan pada bagian E untuk memuat kumpulan dataset terindeks yang sudah diketahui.
- **group_by**: aturan untuk mengelompokkan beberapa pengamatan dalam periode yang sama.
  Nilai yang umum digunakan ialah `"solar_day"`.
- **dask_chunks**: konfigurasi chunk untuk pemuatan tunda dan paralel dengan Dask.
  Parameter ini berguna ketika data terlalu besar untuk dimuat sekaligus ke dalam memori.
- **resampling**: metode untuk menghitung nilai piksel saat reproyeksi atau perubahan resolusi.
  Nilai yang umum digunakan ialah `"nearest"` sebagai nilai bawaan, `"bilinear"`, dan `"cubic"`.

Ekspresi `**query` menguraikan dictionary menjadi argumen bernama untuk `dc.load`.
ODC memakai pengaturan pencarian untuk menemukan dataset terindeks yang cocok, membaca measurement yang diminta, lalu menata piksel pada grid keluaran.
Hasilnya berupa `xarray.Dataset`, yaitu objek Python multidimensi yang dibahas dalam [04_xarray_untuk_open_data_cube.ipynb](04_xarray_untuk_open_data_cube.ipynb).

In [ ]:
ds = dc.load(**query)
ds # ds adalah singkatan umum untuk dataset

## C. Struktur `xarray.Dataset`

Ringkasan yang ditampilkan menjelaskan struktur `xarray.Dataset` hasil pemuatan.
Bagian utamanya juga tersedia sebagai atribut `ds`:

- **Dimensi**: ukuran setiap sumbu data.
  Hasil ini memiliki satu irisan waktu (`time: 1`), 369 baris (`y: 369`), dan 372 kolom (`x: 372`).
- **Koordinat**: label di sepanjang sumbu tersebut.
  `time` memuat timestamp, `y` dan `x` memuat koordinat terproyeksi dalam `output_crs`, sedangkan `spatial_ref` menjelaskan sistem referensi koordinatnya.
- **Variabel data**: measurement yang diminta.
  Setiap variabel berisi array nilai terukur untuk seluruh waktu dan posisi piksel (`y`, `x`) yang dikembalikan.

Setiap measurement berbentuk `xarray.DataArray` dan dapat dipilih berdasarkan namanya.
Ekspresi berikut memilih measurement merah beserta koordinat dan dimensinya:

In [ ]:
ds.red

Plot dapat digunakan untuk memeriksa pola spasial nilai hasil pemuatan secara singkat:

In [ ]:
ds.red.plot()

Sumbu plot menunjukkan koordinat x dan y yang telah diproyeksikan, sedangkan skala warna mewakili nilai measurement merah.

## D. Mencari dataset

Daftar data yang cocok terkadang perlu diperiksa sebelum array piksel dimuat.
`dc.find_datasets` menelusuri indeks datacube dan mengembalikan daftar objek `Dataset` ODC tanpa membaca pikselnya.
Hasil pencarian ini dapat dipakai untuk menghitung jumlah kecocokan, memeriksa metadata, atau memilih subset tertentu bagi `dc.load`.

Objek `Dataset` ODC merupakan catatan metadata untuk satu item data terindeks, termasuk lokasi penyimpanan measurement serta cakupan waktu dan wilayahnya.
Untuk produk dalam contoh ini, setiap catatan yang cocok menjelaskan satu dataset komposit tahunan.
Objek tersebut berbeda dari `xarray.Dataset` berisi array piksel yang dikembalikan oleh `dc.load`.

Pencarian memakai batasan produk, wilayah, dan waktu dari kueri pemuatan.
Pengaturan grid keluaran seperti `output_crs` dan `resolution`, serta pilihan `measurements`, baru diperlukan ketika piksel dimuat.

In [ ]:
datasets = dc.find_datasets(
    product="s2_geomad_annual",
    x=(98.80, 98.90),
    y=(2.65, 2.55),
    time=("2020", "2024"),
)

len(datasets)

`len(datasets)` menunjukkan jumlah dataset terindeks yang sesuai dengan batasan produk, wilayah, dan waktu.
Objek pertama kemudian dapat diperiksa secara langsung:

In [ ]:
datasets[0]

Metadatanya mencakup lokasi file, CRS asal, jalur measurement, dan timestamp.
Pada tahap ini, array measurement belum dibaca.

## E. Memuat dataset terpilih

Daftar hasil `dc.find_datasets` dapat langsung diteruskan kepada `dc.load`.
Parameter `datasets` membuat `dc.load` memakai catatan terindeks yang telah dipilih tanpa mengulangi pencarian katalog.
Measurement dan grid keluaran tetap perlu ditetapkan karena keduanya menentukan array yang dibaca dan susunan hasilnya.

Rentang x dan y pada `dc.find_datasets` menentukan catatan yang cakupan spasialnya berpotongan dengan wilayah pencarian.
Setiap catatan yang cocok tetap membawa cakupan spasial lengkapnya.
Pemanggilan `dc.load` berikut tidak menyertakan batas spasial sehingga ODC dapat memuat seluruh cakupan tersebut dan membutuhkan memori yang jauh lebih besar daripada pemuatan pertama.
Untuk analisis, sertakan batas x dan y pada `dc.load` agar wilayah pemuatan tetap terbatas, atau gunakan `dask_chunks` jika data yang lebih luas memang diperlukan.

In [ ]:
ds = dc.load(
    datasets=datasets,
    measurements=["red", "green", "blue"],
    output_crs="EPSG:32647",
    resolution=(-30, 30),
)

ds

Hasilnya tetap memiliki dimensi `time`, `y`, dan `x` serta variabel data merah, hijau, dan biru, tetapi ukuran dimensi dan nilai koordinatnya dapat berbeda dari pemuatan pertama.
Daftar tersebut berisi komposit tahunan dari beberapa tahun sehingga hasilnya memiliki lebih banyak koordinat waktu.
Ukuran dan nilai koordinat x serta y juga dapat berubah karena cakupan spasial dimuat secara lengkap.

## F. Fungsi bantu geometri

Kueri spasial membutuhkan batas horizontal dan vertikal yang terkadang merepotkan jika dihitung dari suatu lokasi.
Salah satu cara yang umum ialah membentuk kotak pembatas dari titik pusat dan radius.
Komunitas Open Data Cube mengembangkan dan memelihara `odc.geo`, pustaka yang menyediakan fungsi geometri untuk keperluan tersebut.

In [ ]:
from odc.geo.geom import point

lat, lon = -8.65, 115.20  # Denpasar, Bali
bbox = point(lon, lat, crs="EPSG:4326").buffer(0.2).boundingbox
bbox

Fungsi `point` menerima bujur terlebih dahulu, kemudian lintang, lalu menghasilkan objek `Geometry`.
Argumen `crs` menyatakan bahwa koordinat tersebut memakai `EPSG:4326`, yang juga dikenal sebagai WGS84.
Pemanggilan `.buffer(0.2)` membentuk geometri di sekeliling titik dengan jarak 0,2 derajat ke setiap arah, atau kira-kira 22 km di dekat khatulistiwa.
Operasi `.boundingbox` kemudian mengambil batas persegi panjang dari geometri tersebut dan menyimpannya dalam `bbox`.

Kotak pembatas dapat digambar pada peta dasar interaktif dengan `.explore()`.
Tampilan ini membantu memastikan bahwa wilayah hasil perhitungan sudah mengelilingi lokasi yang dimaksud sebelum data dimuat.

In [ ]:
bbox.explore()

Atribut `.left`, `.right`, `.bottom`, dan `.top` menyediakan empat tepi yang diperlukan oleh kueri.
`query.update` mengganti rentang x, rentang y, dan CRS keluaran tanpa mengubah produk, waktu, measurement, serta resolusi yang telah ditetapkan.
Bali berada di UTM zona 50S sehingga `EPSG:32750` menggantikan zona UTM untuk wilayah pertama.

In [ ]:
query.update({
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "output_crs": "EPSG:32750",
})

ds = dc.load(**query)
ds

Measurement merah, hijau, dan biru yang telah dimuat dapat digabungkan menjadi visualisasi RGB singkat:

In [ ]:
ds.to_array().isel(time=0).plot.imshow(vmin=0, vmax=3000)

Plot tersebut memilih koordinat waktu pertama dan memakai batas tampilan 0 serta 3000 agar citra lebih mudah dibaca.
Batas ini hanya memengaruhi visualisasi dan tidak mengubah nilai yang tersimpan dalam `ds`.

## G. Langkah berikutnya

Notebook berikut membahas `xarray.Dataset` hasil `dc.load` secara lebih rinci, termasuk dimensi dan koordinat, pemilihan subset, serta penggabungan variabel.

Materi berikutnya tersedia di [04_xarray_untuk_open_data_cube.ipynb](04_xarray_untuk_open_data_cube.ipynb).